# BBOP Full Preprocessing Pipeline

**Author:** Mir Qi  
**Last Updated:** March 2026  

## Pipeline Overview

This notebook runs the complete preprocessing pipeline for multi-camera behavioral recordings,
including automatic dead-camera detection/rescue and dropped-frame handling.

| Step | Module | Description |
|------|--------|-------------|
| 0 | **Scan & Log** | Index all recording sessions into per-session parquets |
| 1 | **mir_generate_param** | Generate calibration MAT from raw checkerboard data |
| 2 | **Dead-Camera Rescue** | Detect cameras missing `metadata.csv`, clone donor video + calib params |
| 2b | **Rsync Rescued Sessions** | Push locally-rescued video files back to the HPC cluster |
| 3 | **Camera Sync** | Align 6-camera timestamps via LED brightness drops |
| 4 | **Drop-Frame Handler** | Detect & fix inconsistent frame counts across cameras |
| 5 | **COM Prediction** | Coarse 3D center-of-mass localization (GPU, SLURM) |
| 6 | **COM Validation** | Trajectory plots + jump detection |
| 7 | **DANNCE Prediction** | Full 3D pose estimation (GPU, SLURM) |
| 8 | **DANNCE Validation** | Pose quality metrics + visualization |

---

## Prerequisites

- Conda environment: `bbop` (local preprocessing) / `sdannce` (GPU predictions)
- 6-camera recording directories with `videos/Camera{1..6}/`
- Calibration folders (e.g. `calib_before/`) in each session

**Data Structure:**
```
base_folder/
  YYYY_MM_DD/
    session_name/
      videos/
        Camera1/ ... Camera6/   # 0.mp4, frametimes.mat, metadata.csv
      calib_before/             # checkerboard calibration
      *label3d_dannce.mat       # calibration MAT (generated by pipeline)
      folder_log.parquet        # status tracking (generated by scan)
```

**Status codes used in parquets:**
- `0` = needs processing
- `1` = done / no action needed
- `2` = special (e.g. rescued dead camera, old session skipped)
- `3` = failed

---
## 0. Scan & Log

Scans all recording sessions under `base_folder` and creates/updates a `folder_log.parquet` in each session directory. The parquet tracks which pipeline steps have been completed.

**Set `base_folder` here — it is used by every cell below.**

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../'))

from status_fields.status_fields_config_oct3v1_brws_deadcam_260330 import STATUS_FIELDS_CONFIG
from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep

# ── Configuration ──────────────────────────────────────────────────────
base_folder = "/data/big_rim/rsync_dcc_sum/26Jan"  # <-- SET THIS
# base_folder = "/hpc/group/tdunn/Bryan_Rigs/BigOpenField/26Jan"  # cluster path

failed_paths_file = None  # optional .txt with failed session paths
rescan_threshold_days = 0.000001  # set to a large number to skip unchanged sessions

log_folder_to_parquet_sep(
    base_folder, failed_paths_file, STATUS_FIELDS_CONFIG,
    force_rescan_rec_files=[],
    rescan_threshold_days=rescan_threshold_days,
)

---
## Load Session Data

Reads all per-session parquets into a single PyArrow table.  
Each row = one recording session, with columns for every status field.

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

all_df = read_all_parquet_files(base_folder)
print(f"Loaded {all_df.num_rows} sessions")

---
## Status Dashboard

Quick overview of pipeline progress across all sessions, split by single vs. social recordings.

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
import pyarrow.compute as pc

_dash = read_all_parquet_files(base_folder)
_n = _dash.num_rows

# Split single vs social
_has_social = 'social' in _dash.column_names
if _has_social:
    _social = _dash.filter(pc.equal(_dash['social'], '1'))
    _single = _dash.filter(pc.not_equal(_dash['social'], '1'))
else:
    _social, _single = None, _dash

def _show(label, tbl):
    print(f"\n{'='*69}")
    print(f"  {label}: {tbl.num_rows} sessions")
    print(f"{'='*69}")
    _shared = ['mir_generate_param', 'dead_cam', 'sync', 'dropf_handle']
    if 'social' in label.lower():
        _extra = ['com', 'dannce', 'social_pred']
    else:
        _extra = ['com', 'com_vis', 'dannce', 'dannce_vis']
    _fields = _shared + _extra
    print(f"  {'Field':<25} {'0 (todo)':<10} {'1 (done)':<10} {'0.5':<10} {'2 (special)':<12} {'3 (failed)':<10}")
    print(f"  {'-'*77}")
    for f in _fields:
        if f not in tbl.column_names:
            print(f"  {f:<25} -- not in parquet --")
            continue
        col = tbl[f].to_pylist()
        c = {}
        for v in col:
            c[str(v)] = c.get(str(v), 0) + 1
        print(f"  {f:<25} {c.get('0',0):<10} {c.get('1',0):<10} {c.get('0.5',0):<10} {c.get('2',0):<12} {c.get('3',0):<10}")

print(f"Total sessions: {_n}")
_show("SINGLE sessions", _single)
if _social is not None and _social.num_rows > 0:
    _show("SOCIAL sessions", _social)

---
## Filter Sessions

Use PyArrow conditions to select sessions for the next processing step.  
Uncomment / modify conditions as needed. Values are **strings** (`'0'`, `'1'`).

**Common workflows:**
- `mir_generate_param == '0'` → run Step 1
- `dead_cam == '0'` → run Step 2
- `sync == '0', mir_generate_param == '1'` → run Step 3
- `dropf_handle == '0', sync == '1'` → run Step 4
- `com == '0'` → run Step 5
- `com == '1', com_vis == '0'` → run Step 6

In [ ]:
import pyarrow.compute as pc
from functools import reduce

table = all_df

conditions = [
    pc.equal(table['mir_generate_param'], '1'),
    pc.equal(table['sync'], '0'),
    # pc.equal(table['com'], '0'),
    # pc.equal(table['com_vis'], '0'),
    # pc.equal(table['dannce'], '0'),
    # pc.equal(table['dannce_vis'], '0'),
    # pc.equal(table['social'], '1'),
]

filter_mask = reduce(pc.and_, conditions)
for_com = table.filter(filter_mask)
for_com

In [ ]:
# Print filtered session paths
experiment_paths = for_com["rec_path"].to_pylist()
for path in experiment_paths:
    print(path[0] if isinstance(path, list) else path)

---
## Step 1: Generate Calibration Parameters (`mir_generate_param`)

Generates the `*label3d_dannce.mat` calibration file from checkerboard images.  
Re-reads parquets fresh so accidental re-runs are safe (idempotent).

**When:** `mir_generate_param == '0'`  
**Output:** `*label3d_dannce.mat` in each session directory

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
from utlis.exe_engine_utlis.comb_all_exe import sequential_process_and_update_mirgenparam
import pyarrow.compute as pc

_fresh = read_all_parquet_files(base_folder)
_to_process = _fresh.filter(pc.equal(_fresh['mir_generate_param'], '0'))
print(f"Sessions needing mir_generate_param: {_to_process.num_rows}")
for p in _to_process["rec_path"].to_pylist():
    print(p[0] if isinstance(p, list) else p)

# Uncomment to run:
# sequential_process_and_update_mirgenparam(_to_process, base_folder)

---
## Step 2: Dead-Camera Rescue

Automatically detects cameras that died during recording (missing `metadata.csv` in
`videos/Camera{i}/`) and rescues the session by:

1. **Copying donor camera videos** into each missing Camera folder  
   (picks the first live camera as donor — video content is placeholder, but keeps
   the 6-camera file structure intact so downstream steps don't crash)
2. **Cloning calibration parameters** (`K`, `RDistort`, `TDistort`, `r`, `t`) from the
   donor camera in the MAT file
3. **Moving the original MAT** to `prev_calib/` so `find_calib_file()` sees only the
   rescued version

**Status codes for `dead_cam`:**
- `0` = has dead camera(s), not yet rescued
- `1` = all 6 cameras alive (no rescue needed)
- `2` = successfully rescued
- `3` = rescue failed

**Important:** After running rescue locally, you must **rsync the rescued sessions
back to the cluster** before running sync/COM/DANNCE. See Step 2b below.

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
from utlis.exe_engine_utlis.comb_all_exe import sequential_process_and_update_dead_cam_rescue
import pyarrow.compute as pc

_fresh = read_all_parquet_files(base_folder)
_to_rescue = _fresh.filter(pc.equal(_fresh['dead_cam'], '0'))
print(f"Sessions with dead cameras needing rescue: {_to_rescue.num_rows}")
for p in _to_rescue["rec_path"].to_pylist():
    print(p[0] if isinstance(p, list) else p)

# Uncomment to run rescue:
# sequential_process_and_update_dead_cam_rescue(_to_rescue, base_folder)

---
## Step 2b: Rsync Rescued Sessions to Cluster

Dead-camera rescue runs **locally** — it copies donor videos into missing Camera
folders and writes a new MAT file. These new files must be pushed to the HPC cluster
before sync/COM/DANNCE can run.

**Why `--delete`?** The rescue moved the original MAT to `prev_calib/`. Without
`--delete`, the cluster keeps the original MAT at root alongside the rescued one,
causing `find_calib_file()` to see two MATs. `--delete` makes the cluster match
your local state exactly.

**Safe because:** these sessions were blocked at `dead_cam=='0'` — the cluster
never ran sync/COM/DANNCE on them, so there's nothing to protect.

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
import pyarrow.compute as pc

# ── Cluster paths ─────────────────────────────────────────────────────
local_base  = base_folder  # e.g. "/data/big_rim/rsync_dcc_sum/26Jan"
remote_user = "lq53"      # <-- your NetID
remote_host = "dcc-login.oit.duke.edu"
remote_base = "/hpc/group/tdunn/Bryan_Rigs/BigOpenField/26Jan"  # <-- cluster path

_fresh = read_all_parquet_files(base_folder)
_rescued = _fresh.filter(pc.equal(_fresh['dead_cam'], '2'))
print(f"Rescued sessions to push: {_rescued.num_rows}\n")

# Generate one rsync command per rescued session
# --delete removes the original MAT on the cluster (it was moved to prev_calib/ locally)
# --exclude protects parquets (paths differ between local and cluster)
cmds = []
for i in range(_rescued.num_rows):
    d = _rescued['date_folder'][i].as_py()
    r = _rescued['rec_file'][i].as_py()
    local_dir  = f"{local_base}/{d}/{r}/"
    remote_dir = f"{remote_user}@{remote_host}:{remote_base}/{d}/{r}/"
    cmd = (
        f"rsync -avz --delete --progress "
        f"--exclude='folder_log.parquet' --exclude='paret/' "
        f"{local_dir} {remote_dir}"
    )
    cmds.append(cmd)
    print(f"# {d}/{r}")
    print(cmd)
    print()

# Uncomment to run all rsync commands:
# import subprocess
# for cmd in cmds:
#     print(f"Running: {cmd[:80]}...")
#     subprocess.run(cmd, shell=True, check=True)
#     print("Done.\n")

---
## Step 3: Camera Synchronization

Aligns multi-camera timestamps by detecting LED brightness drops in the video streams.
The recording protocol includes **3 brightness drops** (LED switches).

**When:** `sync == '0'` and `mir_generate_param == '1'`  
**Output:** `df_synced_*label3d_dannce.mat` (updated frame indices), `videos/6cam_sync.png`

**Manual supervision required** — check `6cam_sync.png` in each session to verify
clean brightness drops. Initial-frame fluctuations can cause false sync.

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
from utlis.exe_engine_utlis.comb_all_exe import sequential_process_and_update_sync
import pyarrow.compute as pc

_fresh = read_all_parquet_files(base_folder)
_to_sync = _fresh.filter(pc.and_(
    pc.equal(_fresh['sync'], '0'),
    pc.equal(_fresh['mir_generate_param'], '1')
))
print(f"Sessions needing sync: {_to_sync.num_rows}")
for p in _to_sync["rec_path"].to_pylist():
    print(p[0] if isinstance(p, list) else p)

# Uncomment to run:
# sequential_process_and_update_sync(_to_sync, base_folder, max_frames=800)

---
## Step 4: Drop-Frame Handler

USB cameras occasionally drop frames, causing inconsistent frame counts across the 6
cameras.  This step detects and fixes the inconsistency by aligning all cameras to a
standard 30 fps timeline using nearest-neighbor interpolation (`np.searchsorted`).

**Smart processing:** The cell first checks each session for consistency (microsecond
check).  Sessions that are already consistent get marked `'1'` immediately — only
truly inconsistent sessions go through the full alignment.

**When:** `dropf_handle == '0'` and `sync == '1'`  
**Output:** `df_dh_*label3d_dannce.mat` (aligned), original moved to `prev_df_calib/`,
`dropf_handle_log.json` with per-camera diagnostics

In [ ]:
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files
from utlis.exe_engine_utlis.drop_frame_handler import (
    load_frametimes, check_max_shapes_consistency, process_drop_frames,
)
from utlis.exe_engine_utlis.comb_all_exe import _update_dropf_handle_parquet
import pyarrow.compute as pc

_fresh = read_all_parquet_files(base_folder)
_to_fix = _fresh.filter(pc.and_(
    pc.equal(_fresh['dropf_handle'], '0'),
    pc.equal(_fresh['sync'], '1'),
))
_paths = [p[0] if isinstance(p, list) else p for p in _to_fix["rec_path"].to_pylist()]

# Pre-filter: only sessions with actual frame inconsistency
_actually_need = []
_mark_consistent = []
for p in _paths:
    ft = load_frametimes(p)
    if not ft:
        continue
    ok, _ = check_max_shapes_consistency(ft)
    if ok:
        _mark_consistent.append(p)
    else:
        _actually_need.append(p)

print(f"Total dropf_handle=='0': {len(_paths)}")
print(f"  Already consistent (will mark '1'): {len(_mark_consistent)}")
print(f"  Actually need fixing: {len(_actually_need)}")
for p in _actually_need:
    print(f"    {os.path.basename(os.path.dirname(p))}/{os.path.basename(p)}")

# Uncomment to run:
# # Mark consistent sessions as done
# for p in _mark_consistent:
#     row = _to_fix.filter(pc.equal(_to_fix['rec_path'], p))
#     if row.num_rows > 0:
#         d = row['date_folder'].to_pylist()[0]
#         r = row['rec_file'].to_pylist()[0]
#         _update_dropf_handle_parquet(base_folder, d, r, '1')
#
# # Process only the inconsistent ones
# for p in _actually_need:
#     print(f"\nProcessing: {p}")
#     result = process_drop_frames(p)
#     row = _to_fix.filter(pc.equal(_to_fix['rec_path'], p))
#     if row.num_rows > 0:
#         d = row['date_folder'].to_pylist()[0]
#         r = row['rec_file'].to_pylist()[0]
#         if result is True or result is None:
#             _update_dropf_handle_parquet(base_folder, d, r, '1')
#         else:
#             _update_dropf_handle_parquet(base_folder, d, r, '3')

---
## Step 5: COM Prediction (SLURM)

Submits center-of-mass prediction jobs to the HPC cluster via SLURM.  
Requires `sdannce` conda environment and GPU partition.

**Single animal** and **social** variants are provided below.

In [ ]:
# # social com

# from utlis.exe_engine_utlis.comb_all_exe import dispatch_slurm_jobs
# filtered_table=for_com

# dispatch_slurm_jobs(
#     base_path=base_folder,
#     table=filtered_table,
#     slurm_launch_file="/hpc/group/tdunn/lq53/tianqing_pytorch_dannce/dannce_/slurm_launch_predict_social.py",
#     predict_flag="--predict_com",
#     conda_env="sdannce",
#     partition="scavenger-gpu",
#     dry_run=False,
#     max_workers=6,
# )

In [ ]:
# single com

from utlis.exe_engine_utlis.comb_all_exe import dispatch_slurm_jobs
filtered_table = for_com
dispatch_slurm_jobs(
    base_path=base_folder,
    table=filtered_table,
    slurm_launch_file="/hpc/group/tdunn/lq53/tianqing_pytorch_dannce/dannce_/slurm_launch_predict.py",
    predict_flag="--predict_com",
    conda_env="sdannce",
    partition="scavenger-gpu",
    dry_run=False,
    max_workers=6,
)

---
## Step 6: COM Validation

Generates trajectory visualizations and detects tracking jumps.

In [ ]:
# single com vis

from utlis.vis_valid_utlis.com_trag_updated import plot_com_all

for_com_vis = for_com
records = [
    {
        'date_folder': date_folder.as_py(),
        'rec_file': rec_file.as_py()
    }
    for date_folder, rec_file in zip(for_com_vis['date_folder'], for_com_vis['rec_file'])
]

for record in records:
    base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
    print(base_path)
    plot_com_all(base_path, perform_jump_indices=True)

In [ ]:
# # social com vis

# from utlis.vis_valid_utlis.scom_traga_utlis import plot_com_all_social

# for_com_vis = filtered_table
# records = [
#     {
#         'date_folder': date_folder.as_py(),
#         'rec_file': rec_file.as_py()
#     }
#     for date_folder, rec_file in zip(for_com_vis['date_folder'], for_com_vis['rec_file'])
# ]

# for record in records:
#     base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
#     print(base_path)
#     plot_com_all_social(base_path, perform_generate_com_video=True)

---
## Step 7: DANNCE Prediction (SLURM)

Full 3D pose estimation via sDANNCE.  
Optionally uses `bad_com.txt` to skip sessions with known bad COM predictions.

In [ ]:
# single dannce predict

from concurrent.futures import ThreadPoolExecutor
import os

for_dannce = for_com

slurm_launch_file = "/hpc/group/tdunn/lq53/251017_new_dannce_files/slurm_launch_predict.py"

def check_expdir(expdir):
    if not os.path.exists(expdir):
        print(f"Skipping: Experiment directory {expdir} does not exist")
        return None
    return expdir

def run_command(base_path, date_folder, rec_file, partition='scavenger-gpu', dry_run=True):
    expdir_path = os.path.join(base_path, date_folder, rec_file)
    if check_expdir(expdir_path) is None:
        return
    command = f"conda run -n sdannce python {slurm_launch_file} --expdir {expdir_path} --predict_dannce --partition {partition}"
    if dry_run:
        print(f"[DRY-RUN] Command: {command}")
    else:
        print(f"Executing command: {command}")
        os.system(command)

# Optional: skip sessions with known bad COM
txt_file = "/hpc/group/tdunn/Bryan_Rigs/BigOpenField/lumi_novel_object_recog/bad_com.txt"
rel_paths_to_skip = set()
if os.path.isfile(txt_file):
    with open(txt_file, 'r') as f:
        for line in f:
            rel_path = line.strip()
            if rel_path:
                rel_paths_to_skip.add(rel_path)

base_path = base_folder
records = [
    {'date_folder': df.as_py(), 'rec_file': rf.as_py()}
    for df, rf in zip(for_dannce['date_folder'], for_dannce['rec_file'])
]

max_concurrent_jobs = 4
dry_run = False

with ThreadPoolExecutor(max_workers=max_concurrent_jobs) as executor:
    futures = []
    for record in records:
        rel_path = os.path.join(record['date_folder'], record['rec_file'])
        expdir_path = os.path.join(base_path, rel_path)
        if expdir_path in rel_paths_to_skip:
            print(f"Skipping: {rel_path} is in the skip list")
            continue
        futures.append(
            executor.submit(run_command, base_path, record['date_folder'], record['rec_file'], 'scavenger-gpu', dry_run)
        )

---
## Step 8: DANNCE Validation

Runs pose quality checks and generates skeleton visualizations.

In [ ]:
# single dannce validation

from useful_files.sophie_check_dannce_mir_modif import dannce_valid
from concurrent.futures import ProcessPoolExecutor, as_completed

for_dannce_vis = for_com

records = [
    {'date_folder': df.as_py(), 'rec_file': rf.as_py()}
    for df, rf in zip(for_dannce_vis['date_folder'], for_dannce_vis['rec_file'])
]

def process_record(record):
    base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
    print(base_path)
    try:
        dannce_valid(base_path)
    except Exception as e:
        print(f"An error occurred while processing {base_path}: {e}")

with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_record, record) for record in records]
    for future in as_completed(futures):
        pass